## EDGE Model Ver. B

### This notebook will contain a variant of the Rocket Richard Baseline Predictor made in rr1.ipynb, that is, this variant will be using General Skater Stats (GSS) + EDGE Stats -- AKA: EDGE Model Ver. B
#### This is done for the sake of model fine tuning and evaluating performance

In [76]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast
from helpersrr import clear_csv, extractPlayerID, placeToStats, fetchSkaterStats, labelWinners, formatEdgeStats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


import importlib
import helpersrr
importlib.reload(helpersrr)

<module 'helpersrr' from '/Users/Joaquin/Documents/GitHub/nhl-trophy-predictor/notebooks/helpersrr.py'>

In [77]:
#global variables
ranks = False
versionA = True    #version A has no shot details, version B has ALL shot details

client = NHLClient()
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")

first_place = rocket_richard[['szn','winner']]
first_ids = placeToStats(first_place)
second_place = rocket_richard[['szn', 'runner_up']]
second_ids = placeToStats(second_place)
third_place = rocket_richard[['szn', 'finalist']]
third_ids = placeToStats(third_place)

../data/formattedwebscraped/rocketrichard.csv


### Milestone 1: Get EDGE Statistics from the API and place them in data/api/EDGEstats

In [78]:
#DO NOT RUN THIS IF COMMENTED -- essentially places relevant EDGE stats files into the data/api/EDGEstats folder
#edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
#fetchSkaterStats('20212022',csv=True,edge=True)
#for season in edgeSzns:
#    fetchSkaterStats(season, csv=True, edge=True)
#    print(f"done season {season}")

### Milestone 2: Make the new feature set (GSS + EDGE Stats)

In [79]:
#get the playerIds of the top 3
rr2firstids = placeToStats(rocket_richard[['szn','winner']])
rr2secondids = placeToStats(rocket_richard[['szn','runner_up']])
rr2thirdids = placeToStats(rocket_richard[['szn','finalist']])

In [80]:
#split training sets and testing set
edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
trainingSetsB = []
testingSetB = []
if versionA == True:
    trainingSetsA = []
    testingSetA = []

to_drop = ['positionCode','lastName','teamAbbrevs','shGoals','shPoints']    #keep full name instead of last name
for year in edgeSzns:
    if ranks == True:
        if versionA == True:
            modifiedDfA = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=True)
            modifiedDfB = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=False)
        else:
            modifiedDfB = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True, versionA=False)
    else:
        if versionA == True:
            modifiedDfA = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=True)
            modifiedDfB = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=False)
        else:
            modifiedDfB = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True, versionA=False)
    
    #feature set for Ver. B
    feature_dfB = modifiedDfB.drop(columns=to_drop)
    feature_dfB = feature_dfB.dropna()
    feature_dfB.loc[feature_dfB ['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
    feature_dfB.loc[feature_dfB ['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model

    if versionA == True:
        feature_dfA = modifiedDfA.drop(columns=to_drop)
        feature_dfA = feature_dfA.dropna()
        feature_dfA.loc[feature_dfA['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
        feature_dfA.loc[feature_dfA['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model
    
    if year == "20212022":  #2025-2026 will be the testing year
        testingSetB.append(feature_dfB)
        if versionA==True:
            testingSetA.append(feature_dfA)
    else:
        trainingSetsB.append(feature_dfB)
        if versionA == True:
            trainingSetsA.append(feature_dfA)

#print(len(testingSetB), len(trainingSetsB))

masterTrainingB = pd.DataFrame()
for train in trainingSetsB:
    masterTrainingB = pd.concat([masterTrainingB,train])

masterTestingB = pd.DataFrame()
masterTestingB = pd.concat([masterTestingB,testingSetB[0]])


if versionA == True:
    masterTrainingA = masterTrainingB.drop(columns=[
                'Behind the Net Shots',
                        'Beyond Red Line Shots',
                        'Center Point Shots',
                        'Crease Shots',
                        'High Slot Shots',
                        'L Circle Shots',
                        'L Corner Shots',
                        'L Net Side Shots',
                        'L Point Shots',
                        'Low Slot Shots',
                        'Offensive Neutral Zone Shots',
                        'Outside L Shots',
                        'Outside R Shots',
                        'R Circle Shots',
                        'R Corner Shots',
                        'R Net Side Shots',
                        'R Point Shots'
                        ])
    masterTestingA = masterTestingB.drop(columns=[
                'Behind the Net Shots',
                        'Beyond Red Line Shots',
                        'Center Point Shots',
                        'Crease Shots',
                        'High Slot Shots',
                        'L Circle Shots',
                        'L Corner Shots',
                        'L Net Side Shots',
                        'L Point Shots',
                        'Low Slot Shots',
                        'Offensive Neutral Zone Shots',
                        'Outside L Shots',
                        'Outside R Shots',
                        'R Circle Shots',
                        'R Corner Shots',
                        'R Net Side Shots',
                        'R Point Shots'
                        ])

In [81]:
#with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
#    display(masterTestingB) 
#-- works--!

### Milestone 3: Train model

In [82]:
if ranks == False:
    train_x = masterTrainingB.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    train_y = masterTrainingB['rrWinner']
    if versionA == True:
        Atrain_x = masterTrainingA.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
        Atrain_y = masterTrainingA['rrWinner']
else:
    train_x = masterTestingB.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    train_y = masterTestingB['rrRank']
    if versionA == True:
        Atrain_x = masterTrainingA.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
        Atrain_y = masterTrainingA['rrRank']

model1 = LogisticRegression(max_iter=20000, class_weight="balanced", penalty='l2', solver='lbfgs', C=0.01)
model1.fit(train_x,train_y)

#new addition: RandomForest
#model2 = RandomForestClassifier(n_estimators=100)
#model2.fit(train_x,train_y)

#new addition: Gradient Boosting; w/ default parameters, but just written for clarification
#model3 = GradientBoostingClassifier(loss='log_loss', learning_rate=0.1) 
#model3.fit(train_x, train_y)

#new addition: testing EDGE Ver. A vs EDGE Ver.B on different penalties and other hyperparameters
if versionA == True:
    model2 = LogisticRegression(max_iter=20000, class_weight="balanced")
    model2.fit(Atrain_x,Atrain_y)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


### Milestone 4: Test on EDGE + GSS of 2025-2026

In [83]:
#test2025 = masterTesting

if ranks == False:
    test_x = masterTestingB.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    test_y = masterTestingB['rrWinner']
    if versionA == True:
        Atest_x = masterTestingA.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
        Atest_y = masterTestingA['rrWinner']
else:
    test_x = masterTestingB.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    test_y = masterTestingB['rrRank']
    if versionA == True:
        Atest_x = masterTestingA.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
        Atest_y = masterTestingA['rrRank']

pred_y1 = model1.predict(test_x)
predictions = pd.Series(pred_y1)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = masterTestingB.join(predictions)

'''
pred_y2 = model2.predict(test_x)
predictions2 = pd.Series(pred_y2)
predictions2 = predictions2.rename('predictions')
predictions2 = predictions2.to_frame()
results2 = test2025.join(predictions2)
'''

if versionA == True:
    pred_y2 = model2.predict(Atest_x)
    predictions2 = pd.Series(pred_y2)
    predictions2 = predictions2.rename('predictions')
    predictions2 = predictions2.to_frame()
    results2 = masterTestingA.join(predictions2)

In [84]:
#for model 1

if ranks == False:
    show = results.loc[results['predictions'] == 1]
else:
    show = results.loc[results['predictions'] != 0]
    show = show.dropna()
    hyman = results.loc[results['rrRank'] != 0]

#view influences on the model
feature_names = train_x.columns
coefficients = pd.Series(model1.coef_[0],index=feature_names)
print(coefficients.sort_values())
#print(masterTrainingB.shape)

with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
    display(show)  


longShots                      -0.112964
gamesPlayed                    -0.096246
R Circle Shots                 -0.078394
topShotSpeed                   -0.076674
R Point Shots                  -0.064889
L Net Side Shots               -0.044786
L Point Shots                  -0.042533
skatingSpeed                   -0.042009
assists                        -0.041570
penaltyMinutes                 -0.025083
R Corner Shots                 -0.013856
totalDistanceSkated            -0.011085
Outside L Shots                -0.008368
Low Slot Shots                 -0.006731
Center Point Shots             -0.005542
L Corner Shots                 -0.004709
shootsCatches                  -0.004599
highShots                      -0.003394
timeOnIcePerGame               -0.001185
shots                          -0.000988
defensiveZonePctg              -0.000641
offensiveZonePctg              -0.000584
neutralZonePctg                -0.000284
averageTOI                     -0.000020
shootingPct     

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,Behind the Net Shots,Beyond Red Line Shots,Center Point Shots,Crease Shots,High Slot Shots,L Circle Shots,L Corner Shots,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots,averageTOI,rrWinner,predictions
2,75,34,90,1.00000,9,82,40,2,26,8476346,64,115,1.40243,6,25,20212022,0.15267,0,262,Johnny Gaudreau,1114.4878,139.8359,37.934,386.8089,6.0033,23,1,101,11,68,20,0.440687,0.180515,0.378798,3,3,11,5,35,55,3,17,8,63,4,21,11,11,0,8,4,18.574797,0.0,1.0
3,55,30,68,0.53401,11,80,55,1,40,8477934,17,110,1.37500,24,41,20212022,0.19784,0,278,Leon Draisaitl,1340.9500,146.6112,38.435,425.5720,7.1209,14,1,78,12,97,26,0.453497,0.173502,0.373002,0,7,4,6,23,19,0,12,7,91,3,9,22,36,1,35,3,22.349167,0.0,1.0
5,46,44,77,0.56224,10,73,60,2,18,8479318,20,106,1.45205,16,29,20212022,0.17241,0,348,Auston Matthews,1236.5479,153.7889,35.671,404.7509,7.1068,33,1,152,21,105,32,0.483103,0.178901,0.337995,2,6,14,7,59,60,0,10,11,98,2,13,18,33,0,7,8,20.609132,1.0,1.0
7,62,30,75,0.46511,6,82,42,0,68,8479314,57,104,1.26829,12,29,20212022,0.16600,0,253,Matthew Tkachuk,1073.9634,141.0751,33.612,348.3836,5.3727,16,0,69,9,115,25,0.451260,0.179671,0.369069,3,4,2,10,25,20,0,9,8,105,7,3,12,24,0,15,6,17.899390,0.0,1.0
25,44,33,62,0.52000,7,76,40,1,44,8477404,13,84,1.10526,7,22,20212022,0.15151,0,264,Jake Guentzel,1205.8421,139.4979,37.312,401.6312,6.3939,12,0,90,16,133,17,0.464878,0.179347,0.355774,1,2,7,11,45,27,0,4,3,122,1,7,3,18,0,11,2,20.097368,0.0,1.0


findings/conclusions on the above can be found in commitlog.md

In [85]:
#for model 2 -- uses SHAP to visualize predictions (later)
if versionA == True:
    if ranks == False:
        show = results2.loc[results2['predictions'] == 1]
    else:
        show = results2.loc[results2['predictions'] != 0]
        show = show.dropna()
        #hyman = results2.loc[results2['rrRank'] != 0]

    feature_names = Atrain_x.columns
    coefficients = pd.Series(model2.coef_[0],index=feature_names)
    print(coefficients.sort_values())

    with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
        display(show) 



gamesPlayed           -0.458924
longShots             -0.406470
skatingSpeed          -0.203019
topShotSpeed          -0.163087
assists               -0.155537
highShots             -0.084366
penaltyMinutes        -0.062049
ppPoints              -0.051702
shootsCatches         -0.037052
totalDistanceSkated   -0.031255
distanceMaxGame       -0.015998
defensiveZonePctg     -0.003830
offensiveZonePctg     -0.002055
neutralZonePctg       -0.001371
averageTOI             0.000067
shootingPct            0.001109
longGoals              0.002749
timeOnIcePerGame       0.004047
pointsPerGame          0.009378
faceoffWinPct          0.012893
shots                  0.025217
midShots               0.031149
gameWinningGoals       0.038216
otGoals                0.119252
highGoals              0.124173
evGoals                0.183647
plusMinus              0.211077
midGoals               0.220247
ppGoals                0.224003
points                 0.253793
evPoints               0.281757
goals   

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,averageTOI,rrWinner,predictions
2,75,34,90,1.00000,9,82,40,2,26,8476346,64,115,1.40243,6,25,20212022,0.15267,0,262,Johnny Gaudreau,1114.4878,139.8359,37.934,386.8089,6.0033,23,1,101,11,68,20,0.440687,0.180515,0.378798,18.574797,0.0,1.0
5,46,44,77,0.56224,10,73,60,2,18,8479318,20,106,1.45205,16,29,20212022,0.17241,0,348,Auston Matthews,1236.5479,153.7889,35.671,404.7509,7.1068,33,1,152,21,105,32,0.483103,0.178901,0.337995,20.609132,1.0,1.0
